# Part 1

### Configuration & Catalog Setup

In [0]:
STORAGE_ACCOUNT = "deassessmentd06fabcc"
CATALOG = "de_assessment_dev"
RAW_PATH = f"abfss://raw@{'deassessmentd06fabcc'}.dfs.core.windows.net"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG} MANAGED LOCATION '{RAW_PATH}/unity-catalog/dev'")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

### Fetch Shows

In [0]:
import requests, json
import pandas as pd

try:
    response = requests.get("https://api.tvmaze.com/shows", timeout=30)
    response.raise_for_status()
    shows = response.json()
except requests.exceptions.RequestException as e:
    raise Exception(f"Failed to fetch shows from TVMaze API: {e}")

shows_pdf = pd.DataFrame(shows)
# Convert any complex columns to JSON strings
for col_name in shows_pdf.columns:
    if shows_pdf[col_name].dtype == object:
        shows_pdf[col_name] = shows_pdf[col_name].apply(lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x)

shows_df = spark.createDataFrame(shows_pdf)

In [0]:
# print(shows_df.schema)

# shows_df.printSchema()
# shows_df.show(5, truncate=False)
# print(shows_df.count())
# print(shows_df.columns)
print(json.dumps(shows[1], indent=2))
# shows_df.select("id", "name", "network").show(5, truncate=False)

### Write shows to ADLS Gen 2 Storage

In [0]:
# Write json to raw container using the external location name (Unity Catalog routes auth correctly)
shows_df.write.mode("overwrite").json(f"{RAW_PATH}/shows/")

### Save as shows Bronze Delta Tables

In [0]:
# Read back and save as Bronze Delta table
bronze_shows = spark.read.option("inferSchema", "true").json(f"{RAW_PATH}/shows/")

bronze_shows.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("de_assessment_dev.bronze.bronze_shows")

print("bronze_shows created:", bronze_shows.count(), "rows")

### Fetch Episodes & Cast

In [0]:
show_ids = [show["id"] for show in shows[:10]]

# Fetch episodes 
episodes = []
for show_id in show_ids:
    try:
        response = requests.get(f"https://api.tvmaze.com/shows/{show_id}/episodes", timeout=30)
        response.raise_for_status()
        episodes.extend(response.json())
    except requests.exceptions.RequestException as e:
        print(f"Warning: failed to fetch episodes for show {show_id}: {e}")

# Fetch cast
cast = []
for show_id in show_ids:
    try:
        response = requests.get(f"https://api.tvmaze.com/shows/{show_id}/cast", timeout=30)
        response.raise_for_status()
        for member in response.json():
            member["show_id"] = show_id # add show_id to be joined later
            cast.append(member)
    except requests.exceptions.RequestException as e:
        print(f"Warning: failed to fetch cast for show {show_id}: {e}")
# print(show_ids)
# print(episodes[:10])
# print(cast[:10])
print(f"Episodes: {len(episodes)}, Cast: {len(cast)}")

### Write Episodes & Cast to ADLS

In [0]:
# Flatten complex fields for Spark
episodes_pdf = pd.DataFrame(episodes)
for col_name in episodes_pdf.columns:
    if episodes_pdf[col_name].dtype == object:
        episodes_pdf[col_name] = episodes_pdf[col_name].apply(lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x)

cast_pdf = pd.DataFrame(cast)
for col_name in cast_pdf.columns:
    if cast_pdf[col_name].dtype == object:
        cast_pdf[col_name] = cast_pdf[col_name].apply(lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x)

episodes_df = spark.createDataFrame(episodes_pdf)
cast_df = spark.createDataFrame(cast_pdf)

episodes_df.write.mode("overwrite").json(f"{RAW_PATH}/episodes/")
cast_df.write.mode("overwrite").json(f"{RAW_PATH}/cast/")

print(f"Written {episodes_df.count()} episodes and {cast_df.count()} cast members to raw")

### Save Episodes & Cast as Bronze Delta Table

In [0]:
bronze_episodes = spark.read.option("inferSchema", "true").json(f"{RAW_PATH}/episodes/")
bronze_cast = spark.read.option("inferSchema", "true").json(f"{RAW_PATH}/cast/")

bronze_episodes.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("de_assessment_dev.bronze.bronze_episodes")

bronze_cast.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("de_assessment_dev.bronze.bronze_cast")

print("bronze_episodes:", bronze_episodes.count(), "rows")
print("bronze_cast:", bronze_cast.count(), "rows")

### Transfer table ownership to DE-Dev-Team

In [0]:
spark.sql("ALTER TABLE de_assessment_dev.bronze.bronze_shows OWNER TO `DE-Dev-Team`")
spark.sql("ALTER TABLE de_assessment_dev.bronze.bronze_episodes OWNER TO `DE-Dev-Team`")
spark.sql("ALTER TABLE de_assessment_dev.bronze.bronze_cast OWNER TO `DE-Dev-Team`")

### Grant access to the Compliance Team

In [0]:
# Granting select, use catalog/schema to Compliance team
spark.sql("Grant SELECT ON TABLE de_assessment_dev.bronze.bronze_shows TO `Compliance-Team`")
spark.sql("Grant SELECT ON TABLE de_assessment_dev.bronze.bronze_episodes TO `Compliance-Team`")
spark.sql("Grant SELECT ON TABLE de_assessment_dev.bronze.bronze_cast TO `Compliance-Team`")

spark.sql("Grant USE CATALOG ON CATALOG de_assessment_dev TO `Compliance-Team`")
spark.sql("Grant USE SCHEMA ON SCHEMA de_assessment_dev.bronze TO `Compliance-Team`")



### Explanation of schema evolution approach.

Bronze ingests raw TVmaze API payloads without enforcing a schema. Delta writes use mergeSchema=True, so if the API adds new fields in the future they are automatically absorbed into the table schema without any code changes or pipeline failures. No data is ever rejected at this layer. Bronze is append-only and schema-flexible by design. 